In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
x_values = [
    [0.0, 0.0, 1.0, 1.0],
    [1.0, 0.0, 1.0, 1.0],
    [0.0, 1.0, 0.0, 1.0],
    [0.0, 0.0, 0.0, 1.0],
]

w1_values = [0.2, 0.4, 0.6, 0.8]
w2_values = [0.9, 0.7, 0.5, 0.3]

def prepare_inputs(x_values, w1_values, w2_values):
    X = np.asarray(x_values, dtype=float)
    w1 = np.asarray(w1_values, dtype=float).reshape(-1)
    w2 = np.asarray(w2_values, dtype=float).reshape(-1)
    if X.shape != (4, 4):
        raise ValueError("x_values must be a 4x4 collection (x1..x4, each with 4 inputs)")
    if w1.size != 4 or w2.size != 4:
        raise ValueError("w1 and w2 must each contain exactly 4 weights")
    W0 = np.column_stack((w1, w2))
    return X, W0

X, W0 = prepare_inputs(x_values, w1_values, w2_values)
W0

In [ ]:
def neuron_distances(x, W):
    return np.sum((x[:, None] - W) ** 2, axis=0)

def bmu_index(x, W):
    d = neuron_distances(x, W)
    return int(np.argmin(d)), d

def learning_rate(epoch, eta0, tau):
    return eta0 * np.exp(-epoch / tau)

def train_sofm_bmu_only(X, W_init, epochs, eta0, tau):
    W = W_init.copy().astype(float)
    distance_history = []
    winner_history = []
    lr_history = []
    sample_history = []
    W_history = [W.copy()]
    for epoch in range(epochs):
        eta = learning_rate(epoch, eta0, tau)
        for s in range(X.shape[0]):
            x = X[s]
            winner, d = bmu_index(x, W)
            W[:, winner] = W[:, winner] + eta * (x - W[:, winner])
            distance_history.append(d)
            winner_history.append(winner)
            lr_history.append(eta)
            sample_history.append(s)
            W_history.append(W.copy())
    return W, np.array(distance_history), np.array(winner_history), np.array(lr_history), np.array(sample_history), np.array(W_history)

In [ ]:
epochs = 1
eta0 = 0.5
tau = epochs / 2

W_final, distance_history, winner_history, lr_history, sample_history, W_history = train_sofm_bmu_only(
    X=X,
    W_init=W0,
    epochs=epochs,
    eta0=eta0,
    tau=tau,
)
W_final

In [ ]:
initial_distances = np.array([neuron_distances(X[i], W0) for i in range(X.shape[0])])
final_distances = np.array([neuron_distances(X[i], W_final) for i in range(X.shape[0])])

print("Initial squared distances [D(y1), D(y2)] for x1..x4")
print(initial_distances)
print("Final weights (columns: y1, y2)")
print(W_final)
print("Final squared distances [D(y1), D(y2)] for x1..x4")
print(final_distances)
print("Per-iteration distances and weights")
for i in range(distance_history.shape[0]):
    s = sample_history[i] + 1
    d1 = distance_history[i, 0]
    d2 = distance_history[i, 1]
    y1 = np.array2string(W_history[i + 1, :, 0], precision=6, separator=", ")
    y2 = np.array2string(W_history[i + 1, :, 1], precision=6, separator=", ")
    print(f"Iter {i + 1:02d} | x{s} | D(y1)={d1:.8f} | D(y2)={d2:.8f} | y1={y1} | y2={y2}")